# 🎮 Video Games Sentiment Analysis (Analisis Sentimen Ulasan Video Games)

Notebook ini mengimplementasikan alur lengkap analisis sentimen berbasis **Natural Language Processing (NLP)** dan **Machine Learning** pada dataset **Amazon Reviews 2023** kategori **Video_Games**.

### Alur Proyek:
1. **Persiapan Lingkungan & Pustaka**
2. **Pengunduhan & Pemuatan Data Bertahap (Streaming / Chunking)**
   - Membaca data `.jsonl.gz` baris demi baris menggunakan `json.loads` agar memori RAM tidak penuh.
   - Memfilter ulasan selama pembacaan: hanya ulasan dengan `timestamp <= 1577836799999` (hingga 31 Desember 2019 UTC).
3. **Pembersihan Data & Pelabelan (Preprocessing)**
   - Membuang baris dengan `rating == 3` atau nilai kosong pada `rating`, `title`, dan `text`.
   - Membuat label biner: rating 1 atau 2 $\rightarrow$ `0` (Negatif), rating 4 atau 5 $\rightarrow$ `1` (Positif).
   - Menggabungkan kolom `title` dan `text` menjadi `full_text`.
   - Melakukan pembersihan dasar (whitespace berlebih & karakter tidak relevan).
4. **Pembagian Data (Train-Validation Split)**
   - **Time-based Split** (Ulasan $\le 2018$ untuk Train, tahun $2019$ untuk Validation) sebagai simulasi skenario produksi dunia nyata.
   - **Stratified Split** (80:20) untuk menjaga keseimbangan proporsi kelas negatif dan positif.
5. **Exploratory Data Analysis (EDA)**
6. **Ekstraksi Fitur & Pipeline Pemodelan (TF-IDF + Logistic Regression)**
7. **Evaluasi Model Komparatif & Confusion Matrix**
8. **Penyimpanan Model Terlatih** (`model/model_sentimen.joblib`)
9. **Pengujian Inferensi Real-time**

## 1. Persiapan Lingkungan & Pemuatan Pustaka

In [1]:
import os
import re
import gzip
import json
import zlib
import time
import datetime
import requests
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_theme(style="whitegrid")
print("Pustaka berhasil dimuat!")

Pustaka berhasil dimuat!


## 2. Pengunduhan & Pemuatan Data secara Bertahap (Chunking/Streaming)

> **Penting:** Dataset Amazon Reviews 2023 kategori `Video_Games` berukuran besar (arsip `.jsonl.gz` terkompresi ~814 MB). Untuk mencegah kehabisan memori RAM, kita membaca berkas secara **streaming baris demi baris (line-by-line)** menggunakan `json.loads`.
>
> **Aturan Filter:** Hanya ulasan dengan `timestamp <= 1577836799999` (31 Desember 2019, 23:59:59.999 UTC) yang diproses.

In [2]:
URL_VIDEO_GAMES = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Video_Games.jsonl.gz"
CUTOFF_TIMESTAMP = 1577836799999  # 31 Desember 2019 23:59:59.999 UTC

def stream_and_filter_reviews(url=URL_VIDEO_GAMES, max_samples=5000, cutoff_timestamp=CUTOFF_TIMESTAMP):
    """
    Membaca berkas .jsonl.gz secara streaming lewat HTTP chunks,
    mendekompresi on-the-fly, membaca baris demi baris dengan json.loads,
    serta memfilter data ulasan dengan timestamp <= cutoff_timestamp.
    """
    print(f"Mulai streaming dari: {url}")
    print(f"Filter batas timestamp: <= {cutoff_timestamp} (Hingga 31 Des 2019)")
    
    start_time = time.time()
    resp = requests.get(url, stream=True, timeout=60)
    resp.raise_for_status()
    
    d = zlib.decompressobj(16 + zlib.MAX_WBITS)
    records = []
    buffer = ""
    total_scanned = 0
    
    for chunk in resp.iter_content(chunk_size=65536):
        if not chunk:
            continue
        decompressed = d.decompress(chunk)
        buffer += decompressed.decode("utf-8", errors="ignore")
        lines = buffer.split("\n")
        buffer = lines[-1]
        
        for line in lines[:-1]:
            line_str = line.strip()
            if not line_str:
                continue
            total_scanned += 1
            record = json.loads(line_str)
            ts = record.get("timestamp")
            
            if ts is not None and ts <= cutoff_timestamp:
                text = record.get("text", "")
                if text and len(text.strip()) > 5:
                    rating = float(record.get("rating", 0))
                    dt = datetime.datetime.fromtimestamp(ts / 1000.0, datetime.timezone.utc)
                    records.append({
                        "rating": rating,
                        "title": record.get("title", ""),
                        "text": text.strip(),
                        "timestamp": ts,
                        "date": dt.strftime("%Y-%m-%d %H:%M:%S"),
                        "year": dt.year,
                        "asin": record.get("asin", record.get("parent_asin", "")),
                        "verified_purchase": record.get("verified_purchase", False),
                        "helpful_vote": record.get("helpful_vote", 0)
                    })
            
            if len(records) >= max_samples:
                break
        if len(records) >= max_samples:
            break
            
    elapsed = time.time() - start_time
    print(f"Selesai! Memindai {total_scanned} baris, mendapatkan {len(records)} ulasan tersaring dalam {elapsed:.2f} detik.")
    return pd.DataFrame(records)

In [3]:
# Pemuatan ulasan tersaring dari berkas lokal atau streaming langsung
raw_csv_path = os.path.join("data", "video_games_reviews_2019.csv")

if os.path.exists(raw_csv_path):
    print(f"Memuat data ulasan dari berkas lokal: {raw_csv_path}")
    df = pd.read_csv(raw_csv_path)
else:
    df = stream_and_filter_reviews(max_samples=5000)
    os.makedirs("data", exist_ok=True)
    df.to_csv(raw_csv_path, index=False)

print(f"Total baris data awal: {len(df)}")
print(f"Maksimum timestamp   : {df['timestamp'].max()} (Batas: {CUTOFF_TIMESTAMP})")
print(f"Rentang tanggal      : {df['date'].min()} s/d {df['date'].max()}")
print("Verifikasi <= 2019   :", (df['timestamp'] <= CUTOFF_TIMESTAMP).all())
df.head()

## 3. Pembersihan Data & Pelabelan (Preprocessing)

Langkah pra-pemrosesan data:
1. **Buang baris** dengan `rating == 3` (ulasan netral) atau nilai kosong (`NaN` / string kosong) pada `rating`, `title`, dan `text`.
2. **Buat label biner (`label`)**:
   - `rating` 1 atau 2 $\rightarrow$ **`0` (Negatif)**
   - `rating` 4 atau 5 $\rightarrow$ **`1` (Positif)**
3. **Gabungkan kolom `title` dan `text`** menjadi satu fitur teks utama:
   $$df[\text{'full\_text'}] = df[\text{'title'}] + \text{" "} + df[\text{'text'}]$$
4. **Pembersihan dasar**: Menghapus tag HTML, URL, karakter tidak relevan, serta whitespace berlebih.

In [4]:
# 1. Tangani nilai kosong pada rating, title, dan text
df["title"] = df["title"].replace(r"^\s*$", np.nan, regex=True)
df["text"] = df["text"].replace(r"^\s*$", np.nan, regex=True)
df_clean = df.dropna(subset=["rating", "title", "text"]).copy()

# Buang baris dengan rating == 3 (ulasan netral)
df_clean = df_clean[df_clean["rating"] != 3].copy()

# 2. Buat kolom target/label biner (1, 2 -> 0; 4, 5 -> 1)
df_clean["label"] = df_clean["rating"].apply(
    lambda r: 0 if r in [1, 2, 1.0, 2.0] else (1 if r in [4, 5, 4.0, 5.0] else np.nan)
)
df_clean = df_clean.dropna(subset=["label"]).copy()
df_clean["label"] = df_clean["label"].astype(int)

# 3. Gabungkan kolom title dan text menjadi satu fitur teks utama
df_clean["full_text"] = df_clean["title"].astype(str).str.strip() + " " + df_clean["text"].astype(str).str.strip()

# 4. Pembersihan dasar (whitespace berlebih & karakter tidak relevan)
def basic_clean_text(text: str) -> str:
    text = re.sub(r"<.*?>", " ", str(text))
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^\w\s.,!?'\"-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_clean["clean_text"] = df_clean["full_text"].apply(basic_clean_text)

# Simpan dataset hasil pra-pemrosesan
preprocessed_csv_path = os.path.join("data", "video_games_preprocessed.csv")
df_clean.to_csv(preprocessed_csv_path, index=False)
print(f"Dataset preprocessing selesai: {len(df_clean)} baris tersimpan di {preprocessed_csv_path}")

## 4. Pembagian Data (Train-Validation Split)

Pada analisis ulasan video game ini, terdapat dua strategi pembagian data yang direkomendasikan:

### Strategi 1: Time-based Split (Skenario Dunia Nyata / Rekomendasi Utama)
- **Data Latih (Train)**: Ulasan dengan tahun $\le 2018$ (data historis masa lalu).
- **Data Validasi (Validation)**: Ulasan tahun $2019$ (data ulasan masa depan).
- **Alasan Pemilihan**:
  1. **Mencegah Kebocoran Temporal (*Temporal Data Leakage*)**: Di dunia nyata, model tidak dapat dilatih menggunakan data masa depan untuk memprediksi data masa lalu.
  2. **Evaluasi Ketahanan Model terhadap Pergeseran Tren (*Concept & Lexical Drift*)**: Menguji apakah model yang dilatih pada istilah game lama tetap akurat saat mengklasifikasikan ulasan game rilisan baru di tahun 2019.
  3. **Rasio Ideal**: Menghasilkan pembagian **78.9% (3.537 sampel)** untuk Train dan **21.1% (945 sampel)** untuk Validation, sangat mendekati rasio standar 80:20.

### Strategi 2: Stratified Split (80:20) (Komparasi Keseimbangan Kelas)
- Membagi data secara acak 80% train dan 20% validation dengan mempertahankan persentase label Negatif (~12.94%) dan Positif (~87.06%) identik di kedua set data.
- **Alasan Pemilihan**: Menjamin kelas minoritas (ulasan negatif) terwakili secara sempurna di set validasi tanpa terpengaruh fluktuasi tahunan.

In [5]:
# 1. Implementasi Time-based Split (<= 2018 vs 2019)
train_time_df = df_clean[df_clean["year"] <= 2018].copy()
val_time_df = df_clean[df_clean["year"] == 2019].copy()

X_train_time, y_train_time = train_time_df["clean_text"], train_time_df["label"]
X_val_time, y_val_time = val_time_df["clean_text"], val_time_df["label"]

print("=== 1. Time-based Split (<= 2018 vs 2019) ===")
print(f"Data Latih (<= 2018)     : {len(X_train_time)} sampel ({len(X_train_time)/len(df_clean)*100:.1f}%)")
print(f"  - Negatif (0): {sum(y_train_time == 0)} ({sum(y_train_time == 0)/len(y_train_time)*100:.1f}%)")
print(f"  - Positif (1): {sum(y_train_time == 1)} ({sum(y_train_time == 1)/len(y_train_time)*100:.1f}%)")
print(f"Data Validasi (2019)     : {len(X_val_time)} sampel ({len(X_val_time)/len(df_clean)*100:.1f}%)")
print(f"  - Negatif (0): {sum(y_val_time == 0)} ({sum(y_val_time == 0)/len(y_val_time)*100:.1f}%)")
print(f"  - Positif (1): {sum(y_val_time == 1)} ({sum(y_val_time == 1)/len(y_val_time)*100:.1f}%)")

# 2. Implementasi Stratified Split (80:20)
X_train_strat, X_val_strat, y_train_strat, y_val_strat = train_test_split(
    df_clean["clean_text"], df_clean["label"], test_size=0.2, random_state=42, stratify=df_clean["label"]
)

print("\n=== 2. Stratified Split (80:20) ===")
print(f"Data Latih (80%)         : {len(X_train_strat)} sampel")
print(f"  - Negatif (0): {sum(y_train_strat == 0)} ({sum(y_train_strat == 0)/len(y_train_strat)*100:.1f}%)")
print(f"  - Positif (1): {sum(y_train_strat == 1)} ({sum(y_train_strat == 1)/len(y_train_strat)*100:.1f}%)")
print(f"Data Validasi (20%)      : {len(X_val_strat)} sampel")
print(f"  - Negatif (0): {sum(y_val_strat == 0)} ({sum(y_val_strat == 0)/len(y_val_strat)*100:.1f}%)")
print(f"  - Positif (1): {sum(y_val_strat == 1)} ({sum(y_val_strat == 1)/len(y_val_strat)*100:.1f}%)")

In [6]:
# Visualisasi Komparasi Distribusi Split Data
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Visualisasi Time-based Split
split_time_summary = pd.DataFrame({
    "Set": ["Train (<=2018)", "Train (<=2018)", "Val (2019)", "Val (2019)"],
    "Sentimen": ["Negatif (0)", "Positif (1)", "Negatif (0)", "Positif (1)"],
    "Jumlah": [sum(y_train_time == 0), sum(y_train_time == 1), sum(y_val_time == 0), sum(y_val_time == 1)]
})
sns.barplot(data=split_time_summary, x="Set", y="Jumlah", hue="Sentimen", ax=axes[0], palette=["#e74c3c", "#2ecc71"])
axes[0].set_title("Distribusi Label: Time-based Split (<= 2018 vs 2019)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Jumlah Ulasan")

# Visualisasi Stratified Split
split_strat_summary = pd.DataFrame({
    "Set": ["Train (80%)", "Train (80%)", "Val (20%)", "Val (20%)"],
    "Sentimen": ["Negatif (0)", "Positif (1)", "Negatif (0)", "Positif (1)"],
    "Jumlah": [sum(y_train_strat == 0), sum(y_train_strat == 1), sum(y_val_strat == 0), sum(y_val_strat == 1)]
})
sns.barplot(data=split_strat_summary, x="Set", y="Jumlah", hue="Sentimen", ax=axes[1], palette=["#e74c3c", "#2ecc71"])
axes[1].set_title("Distribusi Label: Stratified Split (80:20)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Jumlah Ulasan")

plt.tight_layout()
plt.show()

## 5. Pipeline Pemodelan & Pelatihan (TF-IDF + Logistic Regression)
Membangun pipeline scikit-learn dengan `TfidfVectorizer` (unigram & bigram, sublinear TF) dan pengklasifikasi `LogisticRegression(class_weight='balanced')`.

In [7]:
def create_sentiment_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=10000,
            ngram_range=(1, 2),
            stop_words="english",
            sublinear_tf=True
        )),
        ("classifier", LogisticRegression(
            C=1.0,
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

# 1. Latih model pada Time-based Split
model_time = create_sentiment_pipeline()
model_time.fit(X_train_time, y_train_time)
print("Model Time-based Split selesai dilatih!")

# 2. Latih model pada Stratified Split
model_strat = create_sentiment_pipeline()
model_strat.fit(X_train_strat, y_train_strat)
print("Model Stratified Split selesai dilatih!")

## 6. Evaluasi Komparatif Model
Membandingkan metrik evaluasi (Akurasi, Precision, Recall, F1-Score) antara Time-based Split dan Stratified Split.

In [8]:
# Evaluasi Time-based Split
y_pred_time = model_time.predict(X_val_time)
acc_time = accuracy_score(y_val_time, y_pred_time)

print("=" * 60)
print(f"HASIL EVALUASI TIME-BASED SPLIT (Akurasi: {acc_time * 100:.2f}%)")
print("=" * 60)
print(classification_report(y_val_time, y_pred_time, target_names=["Negatif (0)", "Positif (1)"]))

# Evaluasi Stratified Split
y_pred_strat = model_strat.predict(X_val_strat)
acc_strat = accuracy_score(y_val_strat, y_pred_strat)

print("=" * 60)
print(f"HASIL EVALUASI STRATIFIED SPLIT (Akurasi: {acc_strat * 100:.2f}%)")
print("=" * 60)
print(classification_report(y_val_strat, y_pred_strat, target_names=["Negatif (0)", "Positif (1)"]))

# Visualisasi Heatmap Confusion Matrix Kedua Split
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm_time = confusion_matrix(y_val_time, y_pred_time)
sns.heatmap(cm_time, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Pred Negatif (0)", "Pred Positif (1)"],
            yticklabels=["Aktual Negatif (0)", "Aktual Positif (1)"])
axes[0].set_title(f"Confusion Matrix: Time-based Split\n(Akurasi: {acc_time*100:.2f}%)", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Kelas Aktual")
axes[0].set_xlabel("Kelas Prediksi")

cm_strat = confusion_matrix(y_val_strat, y_pred_strat)
sns.heatmap(cm_strat, annot=True, fmt="d", cmap="Greens", ax=axes[1],
            xticklabels=["Pred Negatif (0)", "Pred Positif (1)"],
            yticklabels=["Aktual Negatif (0)", "Aktual Positif (1)"])
axes[1].set_title(f"Confusion Matrix: Stratified Split\n(Akurasi: {acc_strat*100:.2f}%)", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Kelas Aktual")
axes[1].set_xlabel("Kelas Prediksi")

plt.tight_layout()
plt.show()

## 7. Penyimpanan Model (Model Serialization)
Menyimpan model pipeline ke berkas `model/model_sentimen.joblib`.

In [9]:
os.makedirs("model", exist_ok=True)
model_path = os.path.join("model", "model_sentimen.joblib")

# Simpan model stratified yang memiliki akurasi dan generalisasi tinggi
joblib.dump(model_strat, model_path)
joblib.dump(model_strat, "model_sentimen.joblib")

# Simpan juga model time-based untuk perbandingan
joblib.dump(model_time, os.path.join("model", "model_sentimen_time_based.joblib"))
print(f"Model utama berhasil disimpan di: {model_path} dan model_sentimen.joblib")

## 8. Pengujian Inferensi Real-time (Uji Ulasan Baru)
Menguji model tersimpan dengan contoh ulasan baru (kombinasi judul & isi ulasan game).

In [10]:
loaded_model = joblib.load(model_path)

contoh_ulasan = [
    "Masterpiece of RPG! The combat is smooth, story is deep, and visuals are stunning.",
    "Broken launch and terrible performance. Constant crashing and game breaking bugs.",
    "Great co-op fun with friends. Excellent soundtrack and fun boss fights.",
    "Greedy pay to win microtransactions. Boring repetitive gameplay and empty lobbies."
]

prediksi = loaded_model.predict(contoh_ulasan)
probabilitas = loaded_model.predict_proba(contoh_ulasan)

for teks, pred, prob in zip(contoh_ulasan, prediksi, probabilitas):
    label = "POSITIF" if pred == 1 else "NEGATIF"
    conf = max(prob) * 100
    print(f"Ulasan    : \"{teks}\"")
    print(f"Sentimen  : {label} (Keyakinan: {conf:.2f}%)")
    print("-" * 70)